# NLP Emotion Recognition using HuggingFace and Fine-tuning DistillBERT

In [ ]:
!pip install --upgrade transformers
!pip install datasets
!pip install bertviz
!pip install umap-learn
!pip install huggingface_hub

In [ ]:
import pandas as pd
import numpy as np

from huggingface_hub import list_datasets
# list_datasets gives us all the datasets from hugging face

In [ ]:
from datasets import load_dataset

dataset = load_dataset("dair-ai/emotion")

In [ ]:
dataset

In [ ]:
dataset.set_format(type="pandas")

In [ ]:
dataset["train"]

In [ ]:
df = dataset["train"][:]

In [ ]:
df.head()

In [ ]:
dataset["train"].features

In [ ]:
classes = dataset["train"].features["label"].names

In [ ]:
classes

In [ ]:
df["label_name"] = df["label"].apply(lambda x: classes[x])

In [ ]:
df.head()

# Data Analysis

In [ ]:
import matplotlib.pyplot as plt


In [ ]:
df["label_name"].value_counts()

In [ ]:
df["label_name"].value_counts().plot(kind="bar")

In [ ]:
df["words per tweet"] = df["text"].str.split().apply(len)

In [ ]:
df.boxplot(["words per tweet"], by="label_name", grid=False, figsize=(12,6))

In [ ]:
from transformers import AutoTokenizer

model_ckpt = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [ ]:
text = "I love Aalo paratha. I m using Hugging face"
encoded_text = tokenizer(text)
print(encoded_text)

the first and last tokens are special tokens the first one is known as cls(indicates start of a sentence) and second one is known as separetor token(indicates end of a sentence)

In [ ]:
token = tokenizer.convert_ids_to_tokens(encoded_text.input_ids)

print(token)


In [ ]:
tokenizer.vocab_size, tokenizer.model_max_length

# Tokenization of  emotion dataset

In [ ]:
dataset.reset_format() #now the padas type is removed

In [ ]:
def tokenize(batch):
  return tokenizer(batch["text"], padding=True, truncation=True)

In [ ]:
print(tokenize(dataset["train"][:5]))

In [ ]:
dataset_encoded = dataset.map(tokenize, batched=True, batch_size=None)

In [ ]:
dataset_encoded

## sample model

In [ ]:
text

In [ ]:
inputs = tokenizer(text, return_tensors="pt")


In [ ]:
inputs

In [ ]:
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained(model_ckpt)

In [ ]:
model

In [ ]:
with torch.no_grad():
  outputs = model(**inputs)

last_hidden_states = outputs.last_hidden_state

In [ ]:
last_hidden_states

In [ ]:
last_hidden_states.shape

In [ ]:
from transformers import AutoModelForSequenceClassification

num_labels =len(classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=num_labels).to(device)

In [ ]:
from transformers import TrainingArguments


In [ ]:
batch_size = 64
model_name = "distillbert-finetuned-emotion"

trauning_args = TrainingArguments(
    output_dir = model_name,
    num_train_epochs = 2,
    learning_rate = 2e-5,
    per_device_train_batch_size = batch_size,
    per_device_eval_batch_size = batch_size,
    weight_decay = 0.01,
    push_to_hub = False,
    disable_tqdm = False
    )

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
  labels = pred.label_ids
  preds = pred.predictions.argmax(-1)
  f1 = f1_score(labels, preds, average="weighted")
  acc = accuracy_score(labels, preds)
  return {"accuracy": acc, "f1": f1}

In [ ]:
from transformers import Trainer

In [ ]:
trainer = Trainer(
    model = model,
    args = trauning_args,
    compute_metrics = compute_metrics,
    train_dataset = dataset_encoded["train"],
    eval_dataset = dataset_encoded["validation"]
)

In [ ]:
trainer.train()

In [ ]:
pred_output = trainer.predict(dataset_encoded["test"])

pred_output.metrics

In [ ]:
# for prediction

text ="i hellllllooo"

input_encoded = tokenizer(text, return_tensors="pt").to(device)
with torch.no_grad():
  output = model(**input_encoded)
output

In [ ]:
logits = output.logits
pred = torch.argmax(logits,dim=1).item()
pred, classes[pred]